# Alerts Pipeline — Full Run

Runs the complete pipeline end-to-end in a single notebook.  
Each stage is a separate cell so you can inspect intermediate outputs before proceeding.

| Cell | Stage | What happens |
|------|-------|--------------|
| 0 | Config | Set analysis date, optional store filter |
| 1 | Session | Connect to Snowflake |
| 2 | Data load | OSA + hierarchy joined, cached |
| 3 | Stage 1 — Detect | Anomaly signals (WOW / VS_CHAIN / REPEAT) |
| 4 | Stage 2 — Score | Score signals; apply top_n_per_store filter before enrichment |
| 5 | Stage 3 — Facts | Enrich filtered signals with PPG, DoW, chain drift, repeat history |
| 6 | Stage 4 — Investigate | Generate narratives (rule-based by default) |
| 7 | Build Cards | Assemble final alert cards from score + facts + narrative |
| 8 | Preview | Read through top alert cards |
| 9 | Write | Persist to Snowflake — **run manually when satisfied** |

In [3]:
# ── Date window ────────────────────────────────────────────────────────────────
# Set ANALYSIS_DATE_STR to the last day of the week you want to analyse.
# All other windows are derived automatically.
ANALYSIS_DATE_STR = '2025-06-14'

# ── Client config (None = base defaults; 'eastofengland' = client override) ────
CLIENT = None   # or 'eastofengland'

# ── Optional: restrict to specific stores for a faster test run ─────────────
QC_STORE_IDS = None   # e.g. [242, 301] — set to None for all stores

# ── Threshold overrides (None = use config; values override config) ──────────
MIN_WOW_DROP_OVERRIDE           = None   # e.g. 0.01
MIN_CHAIN_GAP_OVERRIDE          = None   # e.g. 0.03
MIN_LOST_SALES_VALUE_OVERRIDE   = None   # e.g. 200
MIN_REPEAT_ALERT_WEEKS_OVERRIDE = None   # e.g. 4
PROJECTION_WEEKS       = 4      # cost-if-not-fixed projection horizon
DRILLDOWN_TOP_N          = 3      # top-N PPGs in drilldown lists

import datetime
analysis_date   = datetime.date.fromisoformat(ANALYSIS_DATE_STR)
THIS_WEEK_END   = str(analysis_date)
THIS_WEEK_START = str(analysis_date - datetime.timedelta(days=6))
LAST_WEEK_END   = str(analysis_date - datetime.timedelta(days=7))
LAST_WEEK_START = str(analysis_date - datetime.timedelta(days=13))
LOOKBACK_START  = str(analysis_date - datetime.timedelta(days=69))  # ~10 weeks
WMD_LOOKBACK    = str(analysis_date - datetime.timedelta(days=27))  # ~4 weeks (WMD detectors)
RUN_DATE        = str(datetime.date.today())

print(f'Analysis date : {THIS_WEEK_END}')
print(f'Run date      : {RUN_DATE}')
print(f'This week     : {THIS_WEEK_START} – {THIS_WEEK_END}')
print(f'Last week     : {LAST_WEEK_START} – {LAST_WEEK_END}')
print(f'Lookback      : {LOOKBACK_START} – {THIS_WEEK_END}')
print(f'WMD lookback  : {WMD_LOOKBACK} – {THIS_WEEK_END}')
print(f'Client        : {CLIENT or "base defaults"}')
print(f'Store filter  : {QC_STORE_IDS if QC_STORE_IDS else "all stores"}')

Analysis date : 2025-06-14
Run date      : 2026-07-17
This week     : 2025-06-08 – 2025-06-14
Last week     : 2025-06-01 – 2025-06-07
Lookback      : 2025-04-06 – 2025-06-14
WMD lookback  : 2025-05-18 – 2025-06-14
Client        : base defaults
Store filter  : all stores


In [4]:
import sys, os, logging, warnings

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

# ── Suppress noisy Snowflake connector output ─────────────────────────────────
logging.getLogger('snowflake.connector').setLevel(logging.ERROR)
logging.getLogger('snowflake.snowpark').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.pipeline.config.loader import load_config
from store_insight.pipeline.pipeline.session import create_session

cfg     = load_config(CLIENT)
print(f'Config loaded: client={cfg.client_name}')

session = create_session(cfg)   # prints 'Snowflake session created.'

Config loaded: client=default


 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/93a31d5d-1a5d-433f-baa0-138549cd2108/saml2?SAMLRequest=nZJPc9owEMW%2Fikc925Zs808DZChMWlpCaIAcehOWICqy5GplTPrpKxuYSQ%2FJoRePR3q7v6d9O7w7Fyo4CQvS6BEiEUaB0LnhUh9GaLu5D%2FsoAMc0Z8poMUKvAtDdeAisUCWdVO5FP4nflQAX%2BEYaaHMxQpXV1DCQQDUrBFCX0%2FXkYUGTCFMGIKzzOHQt4SA968W5ksZxXddRnUbGHuIEYxzjQexVjeQTeoMoP2aU1jiTG3UrOfs3vYMgMc4ahFd4wupa%2BFnqywg%2BouwuIqBfN5tVuHpcb1Awub1uajRUhbBrYU8yF9unxcUAeAdQkG6Ks6g6hmD8DCP2p7IiAm3qvWJHkZuirJxvHfm%2FeC94rMxB%2BoHNZyNUHiWfPeNfdfl9sM0Ou0U6Vwf1eF4uTtweF%2FOlTVZT9aXOtjr58c3kKHi%2BxZs08c4BKjHXTajOH%2BGkG%2BJeSHobPKBJj5Ju1CHpTxTMfKhSM9dW3py3PqJC5taA2TujldSidTlIWUp4h4eE%2BU%2BWpvtwxxgOSdrvZIOcJwT34ya6BF3Wh7ZG7Pi%2FhjKM37a4ruPSJzSfrYyS%2BWtwb2zB3PsBkoi0J5KH%2B1ZKRcGkmnBuBYAPUilTT61gzm%2B9s5VA

 pip install snowflake-connector-python[secure-local-storage]


In [5]:
from snowflake.snowpark import functions as F

# ── Resolve table names from config ──────────────────────────────────────────
OSA_TABLE       = cfg.tables.osa
HIERARCHY_TABLE = cfg.tables.hierarchy
FEEDBACK_TABLE  = cfg.tables.feedback
OUTPUT_TABLE    = cfg.output.alerts

print(f'OSA table      : {OSA_TABLE}')
print(f'Hierarchy table: {HIERARCHY_TABLE}')
print(f'Feedback table : {FEEDBACK_TABLE}')
print(f'Output table   : {OUTPUT_TABLE}')

osa_raw = (
    session.table(OSA_TABLE)
    .filter(F.col('ANALYSIS_DATE') >= LOOKBACK_START)
    .filter(F.col('ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('STORE_ID'),
        F.col('ITEM_CODE'),
        F.col('ACTUAL_SALES_VOLUME'),
        F.col('LOST_SALES_VOLUME'),
        F.col('LOST_SALES_VALUE'),
        F.col('ANALYSIS_DATE'),
        F.col('DEPARTMENT_ID'),
    )
)

item = (
    session.table(HIERARCHY_TABLE)
    .select(
        F.col('ITEM_CODE'),
        F.col('ITEM_ID'),
        F.col('DEPARTMENT_NAME'),
        F.col('SUBCLASS_NAME'),
        F.col('PRODUCT_PROFILE_GROUP_NAME'),
    )
    .distinct()
)

osa = (
    osa_raw
    .join(item, on=['ITEM_CODE'], how='left')
    .filter(F.col('DEPARTMENT_NAME').is_not_null())
    .cache_result()
)

feedback = (
    session.table(FEEDBACK_TABLE)
    .filter(F.col('ALERT_ANALYSIS_DATE') >= LAST_WEEK_START)
    .filter(F.col('ALERT_ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('STORE_ID'),
        F.col('SITE_NAME'),
        F.col('ALERT_TYPE_NAME'),
        F.col('ALERT_ANALYSIS_DATE'),
        F.col('INTERVENTION_ID'),
        F.col('TICK_AND_FLICK_FLAG'),
    )
    .cache_result()
)

# ── WMD tables (markdown, waste) ──────────────────────────────────────────────
markdown_log = (
    session.table(cfg.tables.markdown_log)
    .filter(F.col('LOCAL_DATE') >= WMD_LOOKBACK)
    .filter(F.col('LOCAL_DATE') <= THIS_WEEK_END)
    .cache_result()
)
markdown_sales = (
    session.table(cfg.tables.markdown_sales)
    .filter(F.col('DATE') >= WMD_LOOKBACK)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .cache_result()
)
wastage = (
    session.table(cfg.tables.wastage)
    .filter(F.col('DATE') >= WMD_LOOKBACK)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .cache_result()
)
item_hierarchy  = session.table(cfg.tables.wmd_hierarchy).cache_result()
store_hierarchy = session.table(cfg.tables.store_hierarchy).cache_result()

print(f'\nOSA rows loaded & cached      : {osa.count():,}')
print(f'Feedback rows loaded & cached : {feedback.count():,}')
print(f'Markdown log    rows          : {markdown_log.count():,}')
print(f'Markdown sales  rows          : {markdown_sales.count():,}')
print(f'Wastage         rows          : {wastage.count():,}')
print(f'Item hierarchy  rows          : {item_hierarchy.count():,}')
print(f'Store hierarchy rows          : {store_hierarchy.count():,}')

OSA table      : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.OSA_V1
Hierarchy table: SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.HIERARCHY_V1
Feedback table : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.FEEDBACK_V1
Output table   : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.ALERTS_PIPELINE


SnowparkSQLException: (1304): 01c5c417-0001-b349-0000-1a490697c8e2: 000904 (42000): SQL compilation error: error line 1 at position 20
invalid identifier 'ITEM_ID'
There are existing quoted column identifiers: []. Please use one of them to reference the column. See more details on Snowflake identifier requirements https://docs.snowflake.com/en/sql-reference/identifiers-syntax

In [ ]:
from store_insight.pipeline.stages._1_detect.detect import detect_signals
from store_insight.pipeline.pipeline.config import DetectorConfig
from collections import Counter

data = osa.filter(F.col('STORE_ID').isin(QC_STORE_IDS)) if QC_STORE_IDS else osa

# ── Build DetectorConfig from config with optional overrides ────────────────
cfg_thresholds = cfg.thresholds
cfg_min_wow_drop           = MIN_WOW_DROP_OVERRIDE or cfg_thresholds.OSA.get('min_wow_drop', 0.01)
cfg_min_chain_gap          = MIN_CHAIN_GAP_OVERRIDE or cfg_thresholds.OSA.get('min_chain_gap', 0.03)
cfg_min_lost_sales_value   = MIN_LOST_SALES_VALUE_OVERRIDE or cfg_thresholds.shared.get('min_lost_sales_value', 200.0)
cfg_min_alert_weeks        = MIN_REPEAT_ALERT_WEEKS_OVERRIDE or cfg_thresholds.OSA.get('min_alert_weeks', 4)

det_cfg = DetectorConfig(
    min_lost_sales_value=cfg_min_lost_sales_value,
    min_wow_drop=cfg_min_wow_drop,
    min_chain_gap=cfg_min_chain_gap,
    min_alert_weeks=cfg_min_alert_weeks,
)

print(f'DetectorConfig: min_wow_drop={det_cfg.min_wow_drop}, min_chain_gap={det_cfg.min_chain_gap}, '
      f'min_lost_sales_value={det_cfg.min_lost_sales_value}, min_alert_weeks={det_cfg.min_alert_weeks}')

signals = detect_signals(
    data,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    store_id=QC_STORE_IDS,
    feedback=feedback,
    markdown_log=markdown_log,
    markdown_sales=markdown_sales,
    wastage=wastage,
    item_hierarchy=item_hierarchy,
    store_hierarchy=store_hierarchy,
    config=det_cfg,
    cfg=cfg,
)

trigger_counts = Counter(tuple(sorted(s['trigger_types'])) for s in signals)
type_counts    = Counter(s['signal_type'] for s in signals)
print(f'\nTotal signals: {len(signals)}')
print('By signal type:')
for stype, count in sorted(type_counts.items()):
    print(f'  {stype:<25} {count}')
print('\nBy trigger combination (top 10):')
for triggers, count in sorted(trigger_counts.items(), key=lambda x: -x[1])[:10]:
    print(f'  {"+".join(triggers):<30} {count}')

print('\nTop 5 signals by LSV:')
for s in sorted(signals, key=lambda x: x.get('this_week_lsv', 0), reverse=True)[:5]:
    print(f"  store={s['store_id']:>4}  dept={s.get('department', s.get('signal_type', '')):<30}  "
          f"type={s['signal_type']:<20}  lsv=£{s.get('this_week_lsv', 0):,.2f}")

ConnectionConfig(warehouse='SNOWFLAKE_LEARNING_WH', database='SNOWFLAKE_LEARNING_DB', schema_name='STORE_INSIGHT', role=None)

In [ ]:
from store_insight.pipeline.stages._1_detect.detect import detect_signals
from store_insight.pipeline.pipeline.config import DetectorConfig
from collections import Counter

data = osa.filter(F.col('STORE_ID') == QC_STORE_ID) if QC_STORE_ID else osa

# ── Build DetectorConfig from config with optional overrides ────────────────
cfg_thresholds = cfg.thresholds
cfg_min_wow_drop           = MIN_WOW_DROP_OVERRIDE or cfg_thresholds.OSA.get('min_wow_drop', 0.01)
cfg_min_chain_gap          = MIN_CHAIN_GAP_OVERRIDE or cfg_thresholds.OSA.get('min_chain_gap', 0.03)
cfg_min_lost_sales_value   = MIN_LOST_SALES_VALUE_OVERRIDE or cfg_thresholds.shared.get('min_lost_sales_value', 200.0)
cfg_min_alert_weeks        = MIN_REPEAT_ALERT_WEEKS_OVERRIDE or cfg_thresholds.OSA.get('min_alert_weeks', 4)

det_cfg = DetectorConfig(
    min_lost_sales_value=cfg_min_lost_sales_value,
    min_wow_drop=cfg_min_wow_drop,
    min_chain_gap=cfg_min_chain_gap,
    min_alert_weeks=cfg_min_alert_weeks,
)

print(f'DetectorConfig: min_wow_drop={det_cfg.min_wow_drop}, min_chain_gap={det_cfg.min_chain_gap}, '
      f'min_lost_sales_value={det_cfg.min_lost_sales_value}, min_alert_weeks={det_cfg.min_alert_weeks}')

signals = detect_signals(
    data,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    feedback=feedback,
    markdown_log=markdown_log,
    markdown_sales=markdown_sales,
    wastage=wastage,
    item_hierarchy=item_hierarchy,
    store_hierarchy=store_hierarchy,
    config=det_cfg,
    cfg=cfg,
)

trigger_counts = Counter(tuple(sorted(s['trigger_types'])) for s in signals)
type_counts    = Counter(s['signal_type'] for s in signals)
print(f'\nTotal signals: {len(signals)}')
print('By signal type:')
for stype, count in sorted(type_counts.items()):
    print(f'  {stype:<25} {count}')
print('\nBy trigger combination (top 10):')
for triggers, count in sorted(trigger_counts.items(), key=lambda x: -x[1])[:10]:
    print(f'  {"+".join(triggers):<30} {count}')

print('\nTop 5 signals by LSV:')
for s in sorted(signals, key=lambda x: x.get('this_week_lsv', 0), reverse=True)[:5]:
    print(f"  store={s['store_id']:>4}  dept={s.get('department', s.get('signal_type', '')):<30}  "
          f"type={s['signal_type']:<20}  lsv=£{s.get('this_week_lsv', 0):,.2f}")

In [ ]:
from store_insight.pipeline.stages._1_detect.detect import detect_signals
from store_insight.pipeline.pipeline.config import DetectorConfig
from collections import Counter

data = osa.filter(F.col('STORE_ID') == QC_STORE_ID) if QC_STORE_ID else osa

# ── Build DetectorConfig from config with optional overrides ────────────────
cfg_thresholds = cfg.get('thresholds', {})
cfg_min_wow_drop           = MIN_WOW_DROP_OVERRIDE or cfg_thresholds.get('OSA', {}).get('min_wow_drop', 0.01)
cfg_min_chain_gap          = MIN_CHAIN_GAP_OVERRIDE or cfg_thresholds.get('OSA', {}).get('min_chain_gap', 0.03)
cfg_min_lost_sales_value   = MIN_LOST_SALES_VALUE_OVERRIDE or cfg_thresholds.get('shared', {}).get('min_lost_sales_value', 200.0)
cfg_min_alert_weeks        = MIN_REPEAT_ALERT_WEEKS_OVERRIDE or cfg_thresholds.get('OSA', {}).get('min_alert_weeks', 4)

det_cfg = DetectorConfig(
    min_lost_sales_value=cfg_min_lost_sales_value,
    min_wow_drop=cfg_min_wow_drop,
    min_chain_gap=cfg_min_chain_gap,
    min_alert_weeks=cfg_min_alert_weeks,
)

print(f'DetectorConfig: min_wow_drop={det_cfg.min_wow_drop}, min_chain_gap={det_cfg.min_chain_gap}, '
      f'min_lost_sales_value={det_cfg.min_lost_sales_value}, min_alert_weeks={det_cfg.min_alert_weeks}')

signals = detect_signals(
    data,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    feedback=feedback,
    markdown_log=markdown_log,
    markdown_sales=markdown_sales,
    wastage=wastage,
    item_hierarchy=item_hierarchy,
    store_hierarchy=store_hierarchy,
    config=det_cfg,
    cfg=cfg,
)

trigger_counts = Counter(tuple(sorted(s['trigger_types'])) for s in signals)
type_counts    = Counter(s['signal_type'] for s in signals)
print(f'\nTotal signals: {len(signals)}')
print('By signal type:')
for stype, count in sorted(type_counts.items()):
    print(f'  {stype:<25} {count}')
print('\nBy trigger combination (top 10):')
for triggers, count in sorted(trigger_counts.items(), key=lambda x: -x[1])[:10]:
    print(f'  {"+".join(triggers):<30} {count}')

print('\nTop 5 signals by LSV:')
for s in sorted(signals, key=lambda x: x.get('this_week_lsv', 0), reverse=True)[:5]:
    print(f"  store={s['store_id']:>4}  dept={s.get('department', s.get('signal_type', '')):<30}  "
          f"type={s['signal_type']:<20}  lsv=£{s.get('this_week_lsv', 0):,.2f}")

DetectorConfig: min_wow_drop=0.01, min_chain_gap=0.03, min_lost_sales_value=200.0, min_alert_weeks=4
[1_detect] TICK_AND_FLICK=13 (chain avg 35.4%)
[1_detect] MARKDOWN_OVERRIDE VS_CHAIN=13 (chain avg 4.2%)
[1_detect] MARKDOWN_OVERRIDE WOW=6
[1_detect] MARKDOWN_SELLTHROUGH VS_CHAIN=3 (chain avg 98.9%)
[1_detect] MARKDOWN_SELLTHROUGH WOW=0
[1_detect] MARKDOWN_SKU EXCESSIVE_OVERRIDE=454 (lookback=4w, min_rate=50%)
[1_detect] MARKDOWN_SKU REPEAT_REQUEST=10903 (lookback=4w, min_weeks=3, min_requests=8)
[1_detect] WASTE VS_CHAIN=26 (chain avg 14.1%)
[1_detect] WASTE WOW=31
[1_detect] 11901 signals — TICK_AND_FLICK=13, VS_CHAIN=26, WOW=31, EXCESSIVE_OVERRIDE=454, REPEAT_REQUEST=10903, REPEAT=321

Total signals: 11901
By signal type:
  COMPLIANCE                13
  MARKDOWN_OVERRIDE         13
  MARKDOWN_SELLTHROUGH      3
  MARKDOWN_SKU              11261
  OSA                       574
  WASTE                     37

By trigger combination (top 10):
  REPEAT_REQUEST                 10807
  

In [ ]:
from store_insight.pipeline.stages._2_score.score import score_all
import statistics

score_all(
    signals,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    osa=osa,
    markdown_log=markdown_log,
    markdown_sales=markdown_sales,
)

scores_raw = [s['score'] for s in signals]
print(f'{len(signals)} signals scored')
if scores_raw:
    print(f'Score range  : £{min(scores_raw):,.0f} – £{max(scores_raw):,.0f}')
    print(f'Median score : £{statistics.median(scores_raw):,.0f}')
    print(f'Non-zero     : {sum(1 for s in scores_raw if s > 0)} / {len(scores_raw)}')

# ── top_n_per_store filter (mirrors pipeline.py) ──────────────────────────────
top_n = cfg.output.top_n_per_store or None
if top_n:
    signals_sorted = sorted(signals, key=lambda s: s['score'], reverse=True)
    store_counts: dict = {}
    filtered: list = []
    for sig in signals_sorted:
        sid = sig['store_id']
        if store_counts.get(sid, 0) < top_n:
            filtered.append(sig)
            store_counts[sid] = store_counts.get(sid, 0) + 1
    print(f'\ntop_n_per_store={top_n}: {len(signals)} → {len(filtered)} signals to enrich')
    signals = filtered

print('\nTop 5 signals by score:')
for s in sorted(signals, key=lambda x: x.get('score', 0), reverse=True)[:5]:
    print(f"  store={s['store_id']:>4}  dept={s.get('department', s.get('signal_type', '')):<30}  "
          f"type={s['signal_type']:<20}  score=£{s.get('score', 0):,.2f}")

In [5]:
from store_insight.pipeline.stages._2_score.score import score_all
import statistics

score_all(
    signals,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    osa=osa,
    markdown_log=markdown_log,
    markdown_sales=markdown_sales,
)

scores_raw = [s['score'] for s in signals]
print(f'{len(signals)} signals scored')
if scores_raw:
    print(f'Score range  : £{min(scores_raw):,.0f} – £{max(scores_raw):,.0f}')
    print(f'Median score : £{statistics.median(scores_raw):,.0f}')
    print(f'Non-zero     : {sum(1 for s in scores_raw if s > 0)} / {len(scores_raw)}')

# ── top_n_per_store filter (mirrors pipeline.py) ──────────────────────────────
top_n = int(cfg.get('output', {}).get('top_n_per_store', 0)) or None
if top_n:
    signals_sorted = sorted(signals, key=lambda s: s['score'], reverse=True)
    store_counts: dict = {}
    filtered: list = []
    for sig in signals_sorted:
        sid = sig['store_id']
        if store_counts.get(sid, 0) < top_n:
            filtered.append(sig)
            store_counts[sid] = store_counts.get(sid, 0) + 1
    print(f'\ntop_n_per_store={top_n}: {len(signals)} → {len(filtered)} signals to enrich')
    signals = filtered

print('\nTop 5 signals by score:')
for s in sorted(signals, key=lambda x: x.get('score', 0), reverse=True)[:5]:
    print(f"  store={s['store_id']:>4}  dept={s.get('department', s.get('signal_type', '')):<30}  "
          f"type={s['signal_type']:<20}  score=£{s.get('score', 0):,.2f}")

[4_score] 11901 signals scored
11901 signals scored
Score range  : £0 – £23,650
Median score : £0
Non-zero     : 1023 / 11901

top_n_per_store=10: 11901 → 1230 signals to enrich

Top 5 signals by score:
  store= 517  dept=MARKDOWN_OVERRIDE               type=MARKDOWN_OVERRIDE     score=£23,650.50
  store= 411  dept=MARKDOWN_OVERRIDE               type=MARKDOWN_OVERRIDE     score=£21,073.58
  store= 407  dept=MARKDOWN_OVERRIDE               type=MARKDOWN_OVERRIDE     score=£18,557.32
  store= 425  dept=MARKDOWN_OVERRIDE               type=MARKDOWN_OVERRIDE     score=£15,820.40
  store= 580  dept=MARKDOWN_OVERRIDE               type=MARKDOWN_OVERRIDE     score=£11,881.97


## Stage 3 — Facts
Enriches each signal with PPG drilldowns, day-of-week pattern, chain drift, and repeat item history.

In [6]:
from store_insight.pipeline.stages._3_facts.facts import enrich_signals

enriched = enrich_signals(
    signals,
    osa,   # always pass full chain osa for chain-level gatherers
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    projection_weeks=PROJECTION_WEEKS,
    drilldown_top_n=DRILLDOWN_TOP_N,
    feedback=feedback,
    markdown_log=markdown_log,
    markdown_sales=markdown_sales,
    wastage=wastage,
    item_hierarchy=item_hierarchy,
    cfg=cfg,
)

# QC checks (OSA-specific keys guarded — non-OSA signals won't have them)
missing_ppg   = sum(1 for s in enriched if not s.get('facts', {}).get('subclass_contributors'))
missing_dow   = sum(1 for s in enriched if not s.get('facts', {}).get('dow_pattern'))
missing_chain = sum(1 for s in enriched if s.get('facts', {}).get('chain_this_week_osa') is None)

type_counts = Counter(s.get('signal_type') for s in enriched)
print(f'{len(enriched)} signals enriched')
print('By signal type: ' + ', '.join(f'{k}={v}' for k, v in sorted(type_counts.items())))
print(f'  No PPG contributors : {missing_ppg}')
print(f'  No DoW pattern      : {missing_dow}')
print(f'  No chain OSA        : {missing_chain}')

# Spot check — highest-LSV OSA signal
osa_signals = [s for s in enriched if s.get('signal_type') == 'OSA']
if osa_signals:
    top = max(osa_signals, key=lambda x: x.get('this_week_lsv', 0))
    f   = top.get('facts', {})
    print(f"\nSpot check — store={top['store_id']}  dept={top['department']}")
    print(f"  score            : £{top.get('score', 0):,.2f}")
    print(f"  worst_day        : {f.get('worst_day')}")
    print(f"  chain_this_wk_osa: {f.get('chain_this_week_osa')}%")
    print(f"  cost_projection  : £{f.get('cost_projection', 0):,.2f}")
    if f.get('subclass_contributors'):
        print(f"  top PPG          : {f['subclass_contributors'][0]['subclass_name']} "
              f"£{f['subclass_contributors'][0]['this_week_lsv']:,.2f}")

[2_facts] 1230 signals enriched
1230 signals enriched
By signal type: COMPLIANCE=13, MARKDOWN_OVERRIDE=13, MARKDOWN_SELLTHROUGH=3, MARKDOWN_SKU=648, OSA=516, WASTE=37
  No PPG contributors : 714
  No DoW pattern      : 714
  No chain OSA        : 714

Spot check — store=239  dept=CIGARETTES
  score            : £3,590.43
  worst_day        : {'day': 'Wednesday', 'dow_num': 4, 'lsv': 725.39}
  chain_this_wk_osa: 91.17%
  cost_projection  : £14,361.72
  top PPG          : KINGSIZE STANDARD £2,646.58


## Stage 4 — Investigate
Uses rule-based templates by default.  
To use Cortex instead, set `engine: cortex` in `_4_investigate/signal_type_investigators.yaml`.

In [7]:
from store_insight.pipeline.stages._4_investigate.investigate import investigate

narratives = [investigate(sig, session=session, cfg=cfg) for sig in enriched]

required = ['issue', 'explanation', 'cost_if_not_fixed', 'recommended_action']
blank_fields = [
    (enriched[i]['store_id'], enriched[i].get('department', enriched[i].get('signal_type', '')), k)
    for i, n in enumerate(narratives)
    for k in required if not n.get(k)
]

print(f'{len(narratives)} narratives generated')
if blank_fields:
    print(f'WARNING — {len(blank_fields)} blank fields:')
    for b in blank_fields[:5]:
        print(f'  store={b[0]}  dept={b[1]}  missing={b[2]}')
else:
    print('All four fields populated for every narrative.')

1230 narratives generated
All four fields populated for every narrative.


## Build Alert Cards
Assembles scored + enriched + investigated signals into final alert card dicts, sorted by score.

In [8]:
import statistics

cards = []
for sig, nar in zip(enriched, narratives):
    section = sig.get('department') or sig.get('signal_type', '')
    cards.append({
        'alert_type':         sig['signal_type'],
        'store_id':           sig['store_id'],
        'section':            section,
        'type':               sig['signal_type'],
        'trigger_types':      sig['trigger_types'],
        'issue':              nar['issue'],
        'explanation':        nar['explanation'],
        'cost_if_not_fixed':  nar['cost_if_not_fixed'],
        'recommended_action': nar['recommended_action'],
        'sort_value':         sig['score'],   # stamped by score_all in Stage 2
        'llm_prompt':         nar.get('llm_prompt', ''),
        'signal_json':        '',
        'facts_json':         '',
    })

cards.sort(key=lambda x: x['sort_value'], reverse=True)
scores = [c['sort_value'] for c in cards]

print(f'{len(cards)} alert cards assembled and sorted')
if scores:
    print(f'Score range  : £{min(scores):,.2f} – £{max(scores):,.2f}')
    print(f'Median score : £{statistics.median(scores):,.2f}')

1230 alert cards assembled and sorted
Score range  : £0.00 – £23,650.50
Median score : £3.55


## Preview — Top Alert Cards
Read through the highest-priority alerts before writing to Snowflake.

In [9]:
PREVIEW_N = 10

print(f"{'Rank':>4}  {'Store':>5}  {'Department':<30}  {'Triggers':<25}  {'Score':>10}")
print(f"{'----':>4}  {'-----':>5}  {'-'*30}  {'-'*25}  {'-'*10}")
for i, c in enumerate(cards[:PREVIEW_N], 1):
    triggers = '+'.join(c['trigger_types'])
    print(f"{i:>4}  {c['store_id']:>5}  {c['section']:<30}  {triggers:<25}  {c['sort_value']:>10,.2f}")

Rank  Store  Department                      Triggers                        Score
----  -----  ------------------------------  -------------------------  ----------
   1    517  MARKDOWN_OVERRIDE               VS_CHAIN                    23,650.50
   2    411  MARKDOWN_OVERRIDE               VS_CHAIN+WOW                21,073.58
   3    407  MARKDOWN_OVERRIDE               VS_CHAIN                    18,557.32
   4    425  MARKDOWN_OVERRIDE               VS_CHAIN                    15,820.40
   5    580  MARKDOWN_OVERRIDE               VS_CHAIN                    11,881.97
   6    234  MARKDOWN_OVERRIDE               VS_CHAIN+WOW                 9,984.77
   7    453  MARKDOWN_OVERRIDE               VS_CHAIN                     9,503.06
   8    238  MARKDOWN_OVERRIDE               VS_CHAIN                     7,541.90
   9    494  MARKDOWN_OVERRIDE               VS_CHAIN+WOW                 7,059.31
  10    242  MARKDOWN_OVERRIDE               VS_CHAIN+WOW                 4,998.65


In [10]:
# ── Print full narrative for any card by rank (1-based) ───────────────────────
RANK = 1

c = cards[RANK - 1]
print('=' * 65)
print(f"RANK {RANK}  |  Store {c['store_id']}  |  {c['section']}  |  {'+'.join(c['trigger_types'])}")
print(f"Score: £{c['sort_value']:,.2f}")
print('=' * 65)
print(f"\nISSUE\n{c['issue']}")
print(f"\nEXPLANATION\n{c['explanation']}")
print(f"\nCOST IF NOT FIXED\n{c['cost_if_not_fixed']}")
print(f"\nRECOMMENDED ACTION\n{c['recommended_action']}")

RANK 1  |  Store 517  |  MARKDOWN_OVERRIDE  |  VS_CHAIN
Score: £23,650.50

ISSUE
Acle Supermarket: markdown override rate 22.4% — 18.2pp above chain average

EXPLANATION
295 of 1318 markdowns were overridden this week (22.4%).
Chain average is 4.2% — this store is 18.2pp above.
Top department: Unknown (295 overrides, 22.4%).

COST IF NOT FIXED
Items overridden at a higher price are at increased risk of reaching end-of-life unsold, driving unnecessary waste.

RECOMMENDED ACTION
Review override decisions in Unknown with the store manager. Ensure markdown guidance is followed — overrides should only be made where there is a clear commercial reason.


## Write to Snowflake

> **Only run this cell when you are satisfied with the output above.**  
> Set `OVERWRITE = True` to replace all rows for this `RUN_DATE` (safe to re-run).  
> Set `OVERWRITE = False` to append (use when running multiple store batches).
>
> Output table is resolved from config (`cfg['output']['alerts']`).

In [11]:
OVERWRITE = False   # set True to overwrite existing rows for this run date

from store_insight.pipeline.pipeline.pipeline import write_alerts
write_alerts(session, cards, run_date=RUN_DATE, overwrite=OVERWRITE, cfg=cfg)

[pipeline] Written 1230 alert cards → SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.ALERTS_PIPELINE (mode=append)
